# Module 01: Basic MIDAS Regression

## Learning Objectives
By completing this notebook, you will:
1. Implement the MIDAS equation step-by-step from scratch
2. Estimate a MIDAS model (GDP ~ Industrial Production) using NLS with Beta polynomial weights
3. Visualize the estimated weight function and compare it to equal-weight aggregation
4. Interpret MIDAS coefficients and R-squared in economic terms

## Prerequisites
- Module 00 notebooks completed (data loaded, MIDAS matrix understood)
- Guide 01: The MIDAS equation
- Guide 02: Weight functions

## Estimated Time: 15 minutes

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize, differential_evolution
from scipy.stats import beta as beta_dist
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

# Data directory
RESOURCES_DIR = '../../module_00_foundations/resources'
if not os.path.exists(RESOURCES_DIR):
    RESOURCES_DIR = '../../../module_00_foundations/resources'

print("Imports complete.")

In [ ]:
section_divider("1. Load Data")

In [ ]:
learning_objectives(["Implement the MIDAS equation step-by-step from scratch", "Estimate a MIDAS model (GDP ~ Industrial Production) using NLS with Beta polynomial weights", "Visualize the estimated weight function and compare it to equal-weight aggregation", "Interpret MIDAS coefficients and R-squared in economic terms"])

In [ ]:
apply_course_theme()
apply_plot_theme()

## 1. Load Data

We load quarterly GDP growth and monthly industrial production growth — the canonical MIDAS example. The goal: predict quarterly GDP growth using monthly IP observations directly, without pre-aggregating IP.

In [ ]:
# Load quarterly GDP growth
gdp = pd.read_csv(
    os.path.join(RESOURCES_DIR, 'gdp_quarterly.csv'),
    index_col='date', parse_dates=True
).squeeze()
gdp.index = pd.PeriodIndex(gdp.index, freq='Q')
gdp.name = 'gdp_growth'

# Load monthly IP growth
ip = pd.read_csv(
    os.path.join(RESOURCES_DIR, 'industrial_production_monthly.csv'),
    index_col='date', parse_dates=True
).squeeze()
ip.index = pd.PeriodIndex(ip.index, freq='M')
ip.name = 'ip_growth'

print(f"GDP: {len(gdp)} quarterly observations ({gdp.index[0]} to {gdp.index[-1]})")
print(f"IP:  {len(ip)} monthly observations ({ip.index[0]} to {ip.index[-1]})")
print(f"\nGDP mean: {gdp.mean():.3f}%  std: {gdp.std():.3f}%")
print(f"IP mean:  {ip.mean():.3f}%  std: {ip.std():.3f}%")

In [ ]:
section_divider("2. Build the MIDAS Data Matrix")

## 2. Build the MIDAS Data Matrix

The MIDAS matrix maps K high-frequency lags to each low-frequency observation. For our setup:
- 4 quarterly lags × 3 months each = K = 12 high-frequency lags
- Each row = one quarter of GDP growth
- Each column = one monthly IP lag

Column 0 is the most recent month (end of current quarter), column 11 is the oldest.

In [ ]:
def build_midas_matrix(y_quarterly, x_monthly, n_quarterly_lags=4, freq_ratio=3):
    """
    Build the MIDAS regressor matrix.
    
    Parameters
    ----------
    y_quarterly : pd.Series with PeriodIndex (freq='Q')
    x_monthly : pd.Series with PeriodIndex (freq='M')
    n_quarterly_lags : int
        Number of quarterly lags (total high-frequency lags = n_quarterly_lags * freq_ratio).
    freq_ratio : int
        High-to-low frequency ratio.
    
    Returns
    -------
    Y : np.ndarray, shape (T,)
    X : np.ndarray, shape (T, K) where K = n_quarterly_lags * freq_ratio
    dates : pd.PeriodIndex
    """
    K = n_quarterly_lags * freq_ratio
    T_Q = len(y_quarterly)
    
    X = np.full((T_Q, K), np.nan)
    
    q_idx = y_quarterly.index
    m_idx = x_monthly.index
    x_vals = x_monthly.values
    
    for t, q in enumerate(q_idx):
        # Last month of quarter q
        last_month_q = q.end_time.to_period('M')
        
        for j in range(K):
            target_month = last_month_q - j
            if target_month in m_idx:
                m_pos = m_idx.get_loc(target_month)
                X[t, j] = x_vals[m_pos]
    
    Y = y_quarterly.values
    valid = ~(np.isnan(X).any(axis=1) | np.isnan(Y))
    
    return Y[valid], X[valid], q_idx[valid]


# Build matrix with 4 quarterly lags
N_Q_LAGS = 4
FREQ_RATIO = 3
K = N_Q_LAGS * FREQ_RATIO  # = 12

Y, X, dates = build_midas_matrix(gdp, ip, n_quarterly_lags=N_Q_LAGS)

print(f"MIDAS matrix built:")
print(f"  Shape: ({len(Y)}, {K})")
print(f"  Sample: {dates[0]} to {dates[-1]}")
print(f"  Column 0 (most recent month): mean={X[:,0].mean():.3f}%")
print(f"  Column 11 (oldest month): mean={X[:,11].mean():.3f}%")

In [ ]:
section_divider("3. Define Weight Functions")

## 3. Define Weight Functions

We implement the two main weight function families: Beta polynomial and Almon polynomial. Both produce normalized, non-negative weights.

In [ ]:
def beta_weights(n_lags, theta1, theta2):
    """
    Beta polynomial MIDAS weights.
    
    theta1, theta2 > 0 control the shape.
    theta2 > theta1: front-loaded (recent lags dominate).
    theta1 = theta2 = 1: uniform (= equal-weight aggregation).
    """
    if theta1 <= 0 or theta2 <= 0:
        return np.ones(n_lags) / n_lags  # Return uniform for invalid params
    
    # Midpoint evaluation: avoids singularities at endpoints when theta < 1
    x = (np.arange(n_lags) + 0.5) / n_lags
    
    # Convention: lag 0 (most recent) maps to large x value
    # So we evaluate Beta PDF at (1 - x) to put mass near small j
    raw = beta_dist.pdf(1 - x, theta1, theta2)
    
    total = raw.sum()
    if total < 1e-12:
        return np.ones(n_lags) / n_lags
    return raw / total


def almon_weights(n_lags, theta1, theta2):
    """
    Exponential Almon polynomial MIDAS weights.
    
    theta1 < 0: declining (typical)
    theta1 < 0, theta2 < 0: hump-shaped
    """
    j = np.arange(n_lags, dtype=float)
    log_raw = theta1 * j + theta2 * j**2
    log_raw -= log_raw.max()  # Numerical stability
    raw = np.exp(log_raw)
    return raw / raw.sum()


# Visualize different weight shapes
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle(f'MIDAS Weight Functions (K={K} lags)', fontsize=12, fontweight='bold')

lags = np.arange(K)

# Beta polynomial shapes
ax = axes[0]
configs_beta = [
    (1.0, 1.0, 'Uniform = equal-weight aggregation', 'gray'),
    (1.0, 5.0, 'Declining (θ₁=1, θ₂=5)', 'steelblue'),
    (1.5, 4.0, 'Typical GDP model (θ₁=1.5, θ₂=4)', 'coral'),
    (2.0, 2.0, 'Bell-shaped (θ₁=2, θ₂=2)', 'green'),
]
for t1, t2, label, color in configs_beta:
    w = beta_weights(K, t1, t2)
    ax.plot(lags, w, 'o-', color=color, linewidth=2, markersize=5, label=label)

for q in [3, 6, 9]:
    ax.axvline(q - 0.5, color='lightgray', linestyle=':', linewidth=1)
ax.text(1.0, ax.get_ylim()[1] * 0.95, 'Curr Q', fontsize=8, color='gray', ha='center')
ax.text(4.0, ax.get_ylim()[1] * 0.95, 'Q-1', fontsize=8, color='gray', ha='center')
ax.text(7.0, ax.get_ylim()[1] * 0.95, 'Q-2', fontsize=8, color='gray', ha='center')
ax.text(10.0, ax.get_ylim()[1] * 0.95, 'Q-3', fontsize=8, color='gray', ha='center')

ax.set_xlabel('Lag j (0 = most recent)')
ax.set_ylabel('Weight')
ax.set_title('Beta Polynomial Family')
ax.legend(fontsize=8, loc='upper right')

# Almon shapes
ax = axes[1]
configs_almon = [
    (0.0, 0.0, 'Uniform (θ₁=0, θ₂=0)', 'gray'),
    (-0.3, 0.0, 'Declining (θ₁=-0.3)', 'steelblue'),
    (-0.1, -0.02, 'Hump (θ₁=-0.1, θ₂=-0.02)', 'coral'),
    (-0.5, 0.0, 'Steep decline (θ₁=-0.5)', 'purple'),
]
for t1, t2, label, color in configs_almon:
    w = almon_weights(K, t1, t2)
    ax.plot(lags, w, 's-', color=color, linewidth=2, markersize=5, label=label)

for q in [3, 6, 9]:
    ax.axvline(q - 0.5, color='lightgray', linestyle=':', linewidth=1)
ax.set_xlabel('Lag j (0 = most recent)')
ax.set_ylabel('Weight')
ax.set_title('Exponential Almon Family')
ax.legend(fontsize=8, loc='upper right')

plt.tight_layout()
plt.savefig('weight_functions.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
section_divider("4. Estimate MIDAS by NLS")

## 4. Estimate MIDAS by NLS

We minimize the sum of squared residuals over $(\alpha, \beta, \theta_1, \theta_2)$. Because the model is nonlinear in $\theta$, we cannot use OLS — instead we use scipy.optimize.minimize.

The objective function:
$$\text{SSE}(\alpha, \beta, \theta) = \sum_{t=1}^T \left(y_t - \alpha - \beta \sum_{j=0}^{K-1} w_j(\theta) x_{mt-j}\right)^2$$

In [ ]:
def midas_sse(params, Y, X, weight_fn='beta'):
    """
    Sum of squared errors for MIDAS model.
    
    Parameters
    ----------
    params : array-like [alpha, beta, theta1, theta2]
    Y : np.ndarray, shape (T,)
    X : np.ndarray, shape (T, K)
    weight_fn : str — 'beta' or 'almon'
    """
    alpha, beta_coef, theta1, theta2 = params
    K = X.shape[1]
    
    if weight_fn == 'beta':
        if theta1 <= 0 or theta2 <= 0:
            return 1e10  # Penalize invalid beta params
        weights = beta_weights(K, theta1, theta2)
    elif weight_fn == 'almon':
        weights = almon_weights(K, theta1, theta2)
    else:
        raise ValueError(f"Unknown weight function: {weight_fn}")
    
    x_weighted = X @ weights  # T-vector: weighted sum of lags for each period
    y_hat = alpha + beta_coef * x_weighted
    residuals = Y - y_hat
    return np.sum(residuals**2)


def estimate_midas_beta(Y, X, n_restarts=5, verbose=True):
    """
    Estimate MIDAS with Beta polynomial weights using NLS.
    
    Uses multiple random starts to avoid local optima.
    
    Returns
    -------
    dict with keys: alpha, beta, theta1, theta2, weights, fitted, residuals, r2
    """
    T, K = X.shape
    
    # Initial estimates from OLS on equal-weight aggregate
    x_mean = X.mean(axis=1)  # Equal-weight aggregate
    ols_beta = np.cov(Y, x_mean)[0, 1] / np.var(x_mean)
    ols_alpha = Y.mean() - ols_beta * x_mean.mean()
    
    # Starting points: try different theta configurations
    starts = [
        [ols_alpha, ols_beta, 1.0, 5.0],   # Declining
        [ols_alpha, ols_beta, 1.5, 4.0],   # Typical
        [ols_alpha, ols_beta, 1.0, 1.0],   # Uniform
        [ols_alpha, ols_beta, 2.0, 3.0],   # Moderate decline
        [ols_alpha, ols_beta, 1.0, 8.0],   # Strong decline
    ]
    
    best_result = None
    best_sse = np.inf
    
    for i, p0 in enumerate(starts[:n_restarts]):
        result = minimize(
            midas_sse,
            p0,
            args=(Y, X, 'beta'),
            method='Nelder-Mead',
            options={'maxiter': 20000, 'xatol': 1e-7, 'fatol': 1e-7, 'adaptive': True}
        )
        
        if result.fun < best_sse:
            best_sse = result.fun
            best_result = result
    
    # Extract parameters from best run
    alpha, beta_coef, theta1, theta2 = best_result.x
    
    # Ensure theta params are positive
    theta1 = max(theta1, 0.01)
    theta2 = max(theta2, 0.01)
    
    weights = beta_weights(K, theta1, theta2)
    x_weighted = X @ weights
    fitted = alpha + beta_coef * x_weighted
    residuals = Y - fitted
    ss_res = np.sum(residuals**2)
    ss_tot = np.sum((Y - Y.mean())**2)
    r2 = 1 - ss_res / ss_tot
    
    if verbose:
        print(f"Beta MIDAS estimates:")
        print(f"  alpha  = {alpha:.4f}")
        print(f"  beta   = {beta_coef:.4f}")
        print(f"  theta1 = {theta1:.4f}")
        print(f"  theta2 = {theta2:.4f}")
        print(f"  R²     = {r2:.4f}")
        print(f"  Converged: {best_result.success}")
    
    return {
        'alpha': alpha, 'beta': beta_coef,
        'theta1': theta1, 'theta2': theta2,
        'weights': weights, 'fitted': fitted,
        'residuals': residuals, 'r2': r2,
        'sse': best_sse, 'converged': best_result.success
    }


print("Estimating MIDAS model (GDP ~ IP, Beta polynomial)...")
print("This takes ~5-10 seconds.")
midas_result = estimate_midas_beta(Y, X)

In [ ]:
section_divider("5. Compare to Baseline Models")

## 5. Compare to Baseline Models

We compare MIDAS to two baselines:
1. **OLS on equal-weight aggregate**: The standard approach (discards timing information)
2. **OLS on last-period**: Using only Month 3 of each quarter

R-squared improvement from MIDAS = value of recovering within-quarter timing information.

In [ ]:
# Baseline 1: OLS on equal-weight aggregate
x_equal = X.mean(axis=1)  # Average of all 12 monthly lags
ols_equal = LinearRegression().fit(x_equal.reshape(-1, 1), Y)
fitted_equal = ols_equal.predict(x_equal.reshape(-1, 1))
r2_equal = r2_score(Y, fitted_equal)
rmse_equal = np.sqrt(np.mean((Y - fitted_equal)**2))

# Baseline 2: OLS on last period (lag 0 only = Month 3 of current quarter)
x_last = X[:, 0]  # Column 0 = most recent month
ols_last = LinearRegression().fit(x_last.reshape(-1, 1), Y)
fitted_last = ols_last.predict(x_last.reshape(-1, 1))
r2_last = r2_score(Y, fitted_last)
rmse_last = np.sqrt(np.mean((Y - fitted_last)**2))

# MIDAS
r2_midas = midas_result['r2']
rmse_midas = np.sqrt(np.mean(midas_result['residuals']**2))

# Naive baseline: mean prediction
fitted_naive = np.full_like(Y, Y.mean())
r2_naive = r2_score(Y, fitted_naive)  # Should be ~0
rmse_naive = np.sqrt(np.mean((Y - fitted_naive)**2))

print("\nModel Comparison: GDP Growth ~ Monthly IP")
print("=" * 55)
print(f"{'Model':<30} {'Params':>6} {'R²':>8} {'RMSE':>8}")
print("-" * 55)
print(f"{'Naive (mean)':<30} {'1':>6} {r2_naive:>8.4f} {rmse_naive:>8.4f}")
print(f"{'OLS (equal-weight avg)':<30} {'2':>6} {r2_equal:>8.4f} {rmse_equal:>8.4f}")
print(f"{'OLS (last-period only)':<30} {'2':>6} {r2_last:>8.4f} {rmse_last:>8.4f}")
print(f"{'Beta MIDAS (K=12)':<30} {'4':>6} {r2_midas:>8.4f} {rmse_midas:>8.4f}")
print()
print(f"MIDAS R² gain over equal-weight: {r2_midas - r2_equal:+.4f}")
print(f"MIDAS R² gain over last-period:  {r2_midas - r2_last:+.4f}")
print(f"MIDAS RMSE improvement over equal-weight: {(rmse_equal - rmse_midas)/rmse_equal:+.1%}")

## 6. Visualize Estimated Weights

The estimated weight function shows which months carry the most information about quarterly GDP growth. A front-loaded pattern (high weight on lag 0) means the most recent month of the quarter dominates. A more spread-out pattern means all months matter approximately equally.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'MIDAS Estimation Results (GDP ~ IP, K={K}, T={len(Y)})',
             fontsize=12, fontweight='bold')

lags = np.arange(K)
q_labels = []
for j in range(K):
    q_off = j // FREQ_RATIO
    m_off = j % FREQ_RATIO
    month_label = ['M3', 'M2', 'M1'][m_off]
    q_name = 'Curr' if q_off == 0 else f'Q-{q_off}'
    q_labels.append(f'{q_name}\n{month_label}')

# Left: Estimated weight function
ax1 = axes[0]
est_weights = midas_result['weights']
equal_weights = np.ones(K) / K

ax1.bar(lags, est_weights, color='steelblue', alpha=0.8, label='Estimated MIDAS weights')
ax1.plot(lags, equal_weights, 'r--', linewidth=2,
         label=f'Equal weight (1/{K}={1/K:.3f})')

# Quarter boundaries
for q in [3, 6, 9]:
    ax1.axvline(q - 0.5, color='gray', linestyle=':', alpha=0.6)
ax1.text(1.0, max(est_weights) * 0.92, 'Curr Q', fontsize=8.5, color='gray', ha='center')
ax1.text(4.0, max(est_weights) * 0.92, 'Q-1', fontsize=8.5, color='gray', ha='center')
ax1.text(7.0, max(est_weights) * 0.92, 'Q-2', fontsize=8.5, color='gray', ha='center')
ax1.text(10.0, max(est_weights) * 0.92, 'Q-3', fontsize=8.5, color='gray', ha='center')

ax1.set_xticks(lags[::2])
ax1.set_xticklabels([q_labels[j] for j in range(0, K, 2)], fontsize=8)
ax1.set_xlabel('Lag (0 = most recent)')
ax1.set_ylabel('Weight')
ax1.set_title(f'Estimated Beta Polynomial Weights\n'
               f'θ₁={midas_result["theta1"]:.2f}, θ₂={midas_result["theta2"]:.2f}')
ax1.legend(fontsize=9)

# Right: Actual vs Fitted
ax2 = axes[1]
t_axis = range(len(Y))
ax2.plot(t_axis, Y, 'o-', color='black', linewidth=1.5, markersize=4, label='Actual GDP growth')
ax2.plot(t_axis, midas_result['fitted'], 's--', color='steelblue',
         linewidth=1.5, markersize=4, label=f'MIDAS fitted (R²={r2_midas:.3f})')
ax2.plot(t_axis, fitted_equal, '^:', color='coral', linewidth=1.2, markersize=3,
         label=f'OLS-aggregate (R²={r2_equal:.3f})', alpha=0.8)

ax2.axhline(0, color='black', linewidth=0.5)
ax2.set_xlabel('Quarter (sample order)')
ax2.set_ylabel('GDP Growth (%)')
ax2.set_title('Actual vs. Fitted: MIDAS vs. OLS-Aggregate')
ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig('midas_estimation.png', dpi=120, bbox_inches='tight')
plt.show()

# Print weight summary
print(f"\nEstimated weight distribution:")
print(f"  Current quarter (lags 0-2):  {est_weights[:3].sum():.1%} of total weight")
print(f"  One quarter back (lags 3-5): {est_weights[3:6].sum():.1%} of total weight")
print(f"  Two quarters back (lags 6-8): {est_weights[6:9].sum():.1%} of total weight")
print(f"  Three quarters back (lags 9-11): {est_weights[9:12].sum():.1%} of total weight")

## 7. Self-Check Questions

Answer these questions by running code and examining results.

In [ ]:
# Self-check 1: Verify the weights sum to 1
weight_sum = midas_result['weights'].sum()
print(f"Weight sum: {weight_sum:.10f}  (should be 1.0)")
assert abs(weight_sum - 1.0) < 1e-8, f"Weights don't sum to 1! Sum = {weight_sum}"
print("  PASS: Weights sum to 1.")

# Self-check 2: MIDAS R² should be at least as good as equal-weight OLS
# (because equal-weight OLS is a special case with theta1=theta2=1)
print(f"\nMIDAS R²: {r2_midas:.4f}")
print(f"OLS R²:   {r2_equal:.4f}")
if r2_midas >= r2_equal - 0.001:  # Small tolerance for NLS convergence
    print("  PASS: MIDAS R² >= OLS R² (in-sample, as expected).")
else:
    print("  NOTE: OLS has higher R². NLS may not have fully converged.")
    print("  Try increasing n_restarts in estimate_midas_beta().")

# Self-check 3: Beta(1,1) should produce approximately equal weights
uniform_beta = beta_weights(K, 1.0, 1.0)
expected_uniform = np.ones(K) / K
max_diff = np.max(np.abs(uniform_beta - expected_uniform))
print(f"\nBeta(1,1) vs equal weights: max difference = {max_diff:.6f}")
assert max_diff < 0.02, f"Beta(1,1) should be approximately uniform, got max diff {max_diff}"
print("  PASS: Beta(1,1) produces approximately equal weights.")

## 8. Economic Interpretation

Translate the statistical results into economic language.

In [ ]:
alpha = midas_result['alpha']
beta = midas_result['beta']
theta1 = midas_result['theta1']
theta2 = midas_result['theta2']
w = midas_result['weights']

print("Economic Interpretation of MIDAS Results")
print("=" * 50)
print()
print(f"Model: GDP_growth_t = {alpha:.4f} + {beta:.4f} * [weighted IP]_t + ε_t")
print()
print(f"Constant (α = {alpha:.4f}):")
print(f"  Expected GDP growth when IP is zero: {alpha:.2f}%")
print()
print(f"IP coefficient (β = {beta:.4f}):")
print(f"  A 1 percentage point increase in the weighted average of monthly IP")
print(f"  is associated with a {beta:.3f} pp increase in quarterly GDP growth.")
print()
print(f"Weight function (Beta θ₁={theta1:.2f}, θ₂={theta2:.2f}):")
print(f"  Shape: {'declining' if theta2 > theta1 else 'bell-shaped or back-loaded'}")
print(f"  Current quarter's IP: {w[:3].sum():.1%} of total informational weight")
print(f"  Previous quarter's IP: {w[3:6].sum():.1%} of total informational weight")
print()
print(f"Model fit:")
print(f"  R² = {midas_result['r2']:.4f}: MIDAS explains {midas_result['r2']*100:.1f}% of quarterly GDP variance")
print(f"  OLS explains {r2_equal*100:.1f}% (MIDAS adds {(midas_result['r2']-r2_equal)*100:.1f}pp)")
print(f"  RMSE = {rmse_midas:.4f}% vs {rmse_equal:.4f}% for OLS")

## Summary

### What You Built
1. A **MIDAS data matrix** aligning 12 monthly IP lags to each quarterly GDP observation
2. **Beta polynomial weight functions** parameterized by $(\theta_1, \theta_2)$
3. An **NLS estimator** that finds optimal $(\alpha, \beta, \theta_1, \theta_2)$ by minimizing SSE
4. A **comparison** showing MIDAS R² gain over equal-weight aggregation

### Key Results
- The estimated weight function is **front-loaded**: current quarter's monthly IP carries the most weight
- MIDAS improves R² over equal-weight OLS by recovering timing information within the quarter
- Beta(1,1) reduces to equal-weight aggregation — the null hypothesis we can test against

### What's Next
Notebook 02 compares all weight function families (Beta, Almon, U-MIDAS) on the same data, and Notebook 03 visualizes the estimated lag polynomials interactively.

---
*Next: Module 01, Notebook 02 — Weight Function Comparison*

In [ ]:
key_takeaways(["Beta polynomial weight functions"])